# iprPy surface_energy_static calculation

In [1]:
# Standard library imports
import datetime
from copy import deepcopy

# http://www.numpy.org/
import numpy as np

# https://ipython.org/
from IPython.display import display, Code, Markdown, Pretty

# https://github.com/usnistgov/atomman
import atomman as am
import atomman.unitconvert as uc

# https://github.com/usnistgov/iprPy
import iprPy

print('Notebook last executed on', datetime.date.today(), 'using iprPy version', iprPy.__version__)

Notebook last executed on 2026-06-26 using iprPy version 0.12.a


## 1. Load calculation and view description

### 1.1. Load the calculation

In [2]:
# Load the calculation being demoed
calculation = iprPy.load_calculation('surface_energy_static')

### 1.2. Display calculation description and theory

In [3]:
# Display main docs and theory
display(Markdown(calculation.maindoc))
display(Markdown(calculation.theorydoc))

# surface_energy_static calculation style

**Lucas M. Hale**, [lucas.hale@nist.gov](mailto:lucas.hale@nist.gov?Subject=ipr-demo), *Materials Science and Engineering Division, NIST*.

## Introduction

The surface_energy_static calculation style evaluates the formation energy for a free surface by slicing an atomic system along a specific plane.

### Version notes

- 2018-07-09: Notebook added.
- 2019-07-30: Description updated and small changes due to iprPy version.
- v0.10.0: Version 0.10 update - potentials now loaded from database.
- 2020-09-22: Calculation updated to use atomman.defect.FreeSurface class. Setup and parameter definition streamlined.
- v0.11.0: Notebook updated to reflect version 0.11.
- v0.12.0: Method updated to support the LAMMPS library interface.
  
### Additional dependencies

### Disclaimers

- [NIST disclaimers](http://www.nist.gov/public_affairs/disclaimer.cfm)
- Other atomic configurations at the free surface for certain planar cuts may have lower energies.  The atomic relaxation will find a local minimum, which may not be the global minimum.  Additionally, the material cut is planar perfect and therefore does not explore the effects of atomic roughness.


## Method and Theory

First, an initial system is generated.  This is accomplished by

1. Starting with a unit cell system.

2. Generating a transformed system by rotating the unit cell such that the new system's box vectors correspond to crystallographic directions, and filled in with atoms to remain a perfect bulk cell when the three boundaries are periodic.

3. All atoms are shifted by a fractional amount of the box vectors if needed.

4. A supercell system is constructed by combining multiple replicas of the transformed system.

Two LAMMPS simulations are then performed that apply an energy/force minimization on the system, and the total energy of the system after relaxing is measured, $E_{total}$.  In the first simulation, all of the box's directions are kept periodic (ppp), while in the second simulation two are periodic and one is non-periodic (ppm).  This effectively slices the system along the boundary plane creating two free surfaces, each with surface area

$$A = \left| \vec{a_1} \times \vec{a_2} \right|,$$

where $\vec{a_1}$ and $\vec{a_2}$ are the two lattice vectors corresponding to the periodic in-plane directions.

The formation energy of the free surface, $E_{f}^{surf}$, is computed in units of energy over area as

$$E_{f}^{surf} = \frac{E_{total}^{surf} - E_{total}^{0}} {2 A}.$$

The calculation method allows for the specification of which of the three box dimensions the cut is made along.  If not specified, the default behavior is to make the $\vec{c}$ vector direction non-periodic.  This choice is due to the limitations of how LAMMPS defines triclinic boxes.  $\vec{c}$ vector is the only box vector that is allowed to have a component in the Cartesian z direction.  Because of this, the other two box vectors are normal to the z-axis and therefore will be in the cut plane.


## 2. Display the underlying code

This section displays the underlying code used when calculation.calc() is called in Section 4. It is provided here allowing any users to see and understand how the calculation works.

Feel free to modify and test the functions for yourself!  You can do this by
1. Copy the Python code displayed here into a "Code" cell and run it.
2. In Section 2.2, set "savefiles = True" and run the cell to save any supporting non-python files to the working directory.
3. In Section 4, call the primary calculation function directly rather than using calculation.calc().

### 2.1. Show code and supporting file names and content

In [4]:
# Display calculation code and supporting files
for filename, contents in calculation.files.items():
    display(Markdown(f'## Contents of file "{filename}"'))
    if filename[-3:] == '.py':
        display(Code(contents, language='python'))
    else:
        display(Pretty(contents))

## Contents of file "surface_energy_static.py"

# Python script created by Lucas Hale and Norman Luu.

# Standard library imports
import shutil
from typing import Optional, Union
from pathlib import Path

# http://www.numpy.org/
import numpy as np

# https://github.com/usnistgov/atomman 
import atomman as am
import atomman.unitconvert as uc
from atomman.typing import lammpspotential, unitfloat, lammps, millerindices
from atomman.lammps import LAMMPS, LAMMPSobj

def surface_energy_static(lammps_command: Union[str, LAMMPSobj],
                          ucell: am.System,
                          potential: lammpspotential,
                          hkl: millerindices,
                          mpi_command: Optional[str] = None,
                          sizemults: Union[list, tuple, None] = None,
                          minwidth: Optional[unitfloat] = None,
                          even: bool = False,
                          conventional_setting: str = 'p',
                          cutboxvector: str = 'c',
                          atomshift: Union[list, np.ndarray, None] = None,
                          shiftindex: Optional[int] = None,
                          etol: float = 0.0,
                          ftol: unitfloat = 0.0,
                          maxiter: int = 10000,
                          maxeval: int = 100000,
                          dmax: unitfloat = '0.01 angstrom',
                          usefiles: bool = False) -> dict:
    """
    Evaluates surface formation energies by slicing along one periodic
    boundary of a bulk system.
    
    Parameters
    ----------
    lammps_command : str, LAMMPSEXE or LAMMPSLIB
        LAMMPS executable command, LAMMPS library name, or an atomman LAMMPS
        interface object.
    ucell : atomman.System
        The crystal unit cell to use as the basis of the stacking fault
        configurations.
    potential : PotentialLAMMPS or PotentialLAMMPSKIM
        The LAMMPS implemented potential to use.
    hkl : array-like object or str
        The Miller(-Bravais) crystal fault plane relative to ucell.
    mpi_command : str, optional
        The MPI command for running LAMMPS in parallel.  If not given, LAMMPS
        will run serially.
    sizemults : list or tuple, optional
        The three System.supersize multipliers [a_mult, b_mult, c_mult] to use on the
        rotated cell to build the final system. Note that the cutboxvector sizemult
        must be an integer and not a tuple.  Default value is [1, 1, 1].
    minwidth : float or str, optional
        If given, the sizemult along the cutboxvector will be selected such that the
        width of the resulting final system in that direction will be at least this
        value. If both sizemults and minwidth are given, then the larger of the two
        in the cutboxvector direction will be used. 
    even : bool, optional
        A True value means that the sizemult for cutboxvector will be made an even
        number by adding 1 if it is odd.  Default value is False.
    conventional_setting : str, optional
        Allows for rotations of a primitive unit cell to be determined from
        (hkl) indices specified relative to a conventional unit cell.  Allowed
        settings: 'p' for primitive (no conversion), 'f' for face-centered,
        'i' for body-centered, and 'a', 'b', or 'c' for side-centered.  Default
        behavior is to perform no conversion, i.e. take (hkl) relative to the
        given ucell.
    cutboxvector : str, optional
        Indicates which of the three system box vectors, 'a', 'b', or 'c', to
        cut with a non-periodic boundary (default is 'c').
    atomshift : array-like object, optional
        A Cartesian vector shift to apply to all atoms.  Can be used to shift
        atoms perpendicular to the fault plane to allow different termination
        planes to be cut.  Cannot be given with shiftindex.
    shiftindex : int, optional
        Allows for selection of different termination planes based on the
        preferred shift v

### 2.2. Optional: Save supporting files

Set "savefiles = True" to save files locally.  

Note that the code above should be using atomman.tools.read_calc_file() to read the files, which will read any local files with matching names if they exist or read the packaged version if the local files do not exist. This means that if you save the files locally, you can modify them and see how it affects the calculation!

In [5]:
savefiles = False

if savefiles:
    for filename, contents in calculation.files.items():
        if filename[-3:] != '.py':
            with open(filename, 'w') as f:
                f.write(contents)

## 3. Specify input parameters

### 3.1. System-specific paths

- __lammps_command__ is the LAMMPS command to use (required).
- __mpi_command__ MPI command for running LAMMPS in parallel. A value of None will run simulations serially.

In [6]:
lammps_command = 'lmp_serial'
mpi_command = None
#mpi_command = 'mpiexec -localonly 4'

# Optional: check that LAMMPS works and show its version 
print(f'LAMMPS version = {am.lammps.checkversion(lammps_command)["version"]}')

LAMMPS version = 3 Mar 2020


### 3.2. Interatomic potential

- __potential_name__ gives the name of a potential_LAMMPS record to find and download from the iprPy library.  
- __potential__ is a potential_LAMMPS or potential_LAMMPS_KIM record object (required).

See documentation for the [potentials package](https://github.com/usnistgov/potentials/tree/master/doc) for more options on finding, loading and building potential objects (doc Notebook #s 0, 5.3, 5.4 and 7).

In [7]:
potential_name = '1999--Mishin-Y--Ni--LAMMPS--ipr1'

# Retrieve potential and parameter file(s) using atomman
potential = am.load_lammps_potential(id=potential_name, getfiles=True)

### 3.3. Initial unit cell system

- __ucell__ is an atomman.System representing a fundamental unit cell of the system (required).  Here, this is generated by loading the relaxed fcc crystal for the potential from the database.

See documentation for the [atomman package](https://github.com/lmhale99/atomman/tree/master/doc/tutorial) for more options on building and loading atomic configurations (doc Notebook #s 1.1, 1.2, 1.3, 1.4 and 1.4.*)

In [8]:
# Create ucell by loading prototype record
ucell = am.load('crystal', potential=potential, family='A1--Cu--fcc')

print(ucell)

Multiple matching record retrieved from remote
#  family               symbols  alat    Ecoh    method  standing
 1 A1--Cu--fcc          Ni        3.5200 -4.4500 dynamic good
 2 A1--Cu--fcc          Ni        7.3760  0.0119 dynamic good


Please select one: 1


avect =  [ 3.520,  0.000,  0.000]
bvect =  [ 0.000,  3.520,  0.000]
cvect =  [ 0.000,  0.000,  3.520]
origin = [ 0.000,  0.000,  0.000]
natoms = 4
natypes = 1
symbols = ('Ni',)
pbc = [ True  True  True]
per-atom properties = ['atype', 'pos']
     id |   atype |  pos[0] |  pos[1] |  pos[2]
      0 |       1 |   0.000 |   0.000 |   0.000
      1 |       1 |   0.000 |   1.760 |   1.760
      2 |       1 |   1.760 |   0.000 |   1.760
      3 |       1 |   1.760 |   1.760 |   0.000


### 3.4. Defect parameters

- __hkl__ gives the Miller (hkl) or Miller-Bravais (hkil) plane to create the free surface on.
- __cutboxvector__ specifies which of the three box vectors ('a', 'b', or 'c') is to be made non-periodic to create the free surface. 
- __shiftindex__ can be used for complex crystals to specify different termination planes.

In [9]:
hkl = [1,0,0]
cutboxvector = 'c'
shiftindex = 0

### 3.5. System modifications

- __sizemults__  list of three integers specifying how many times the ucell vectors of $a$, $b$ and $c$ are replicated in creating system.
- __minwidth__ specifies a minimum width that the system should be along the cutboxvector direction. The given sizemult in that direction will be increased if needed to ensure that the system is at least this wide. 

In [10]:
sizemults = [5, 5, 10]
minwidth = uc.set_in_units(0.0, 'angstrom')

### 3.5. Calculation-specific parameters

- __energytolerance__ is the energy tolerance to use during the minimizations. This is unitless.
- __forcetolerance__ is the force tolerance to use during the minimizations. This is in energy/length units.
- __maxiterations__ is the maximum number of minimization iterations to use.
- __maxevaluations__ is the maximum number of minimization evaluations to use.
- __maxatommotion__ is the largest distance that an atom is allowed to move during a minimization iteration. This is in length units.

In [11]:
energytolerance = 1e-8
forcetolerance = uc.set_in_units(0.0, 'eV/angstrom')
maxiterations = 10000
maxevaluations = 100000
maxatommotion = uc.set_in_units(0.01, 'angstrom')

## 4. Run calculation and view results

### 4.1. Run calculation

All primary calculation method functions take a series of inputs and return a dictionary of outputs.

In [12]:
# What is calculation.calc an alias of?
calculation.calc.__module__

'iprPy.calculation.surface_energy_static.surface_energy_static'

In [13]:
results_dict = calculation.calc(lammps_command, ucell, potential, hkl,
                                mpi_command = mpi_command,
                                sizemults = sizemults,
                                minwidth = minwidth,
                                cutboxvector = cutboxvector,
                                shiftindex = shiftindex,
                                etol = energytolerance,
                                ftol = forcetolerance,
                                maxiter = maxiterations,
                                maxeval = maxevaluations,
                                dmax = maxatommotion)
print(results_dict.keys())

dict_keys(['dumpfile_base', 'dumpfile_surf', 'E_total_base', 'E_total_surf', 'A_surf', 'E_pot', 'E_surf_f'])


### 4.2. Report results

Values returned in the results_dict:

- **'dumpfile_base'** (*str*) - The filename of the LAMMPS dump file
  of the relaxed bulk system.
- **'dumpfile_surf'** (*str*) - The filename of the LAMMPS dump file
  of the relaxed system containing the free surfaces.
- **'E_total_base'** (*float*) - The total potential energy of the
  relaxed bulk system.
- **'E_total_surf'** (*float*) - The total potential energy of the
  relaxed system containing the free surfaces.
- **'A_surf'** (*float*) - The area of the free surface.
- **'E_pot'** (*float*) - The per-atom potential energy of the relaxed bulk
  system.
- **'E_surf_f'** (*float*) - The computed surface formation energy.

In [14]:
energy_unit = 'eV'
area_unit = 'nm^2'
energy_area_unit = 'mJ/m^2'

print('E_pot =  ', uc.get_in_units(results_dict['E_pot'], energy_unit), energy_unit)
print('A_surface =', uc.get_in_units(results_dict['A_surf'], area_unit), area_unit)
print('E_surface_f =', uc.get_in_units(results_dict['E_surf_f'], energy_area_unit), energy_area_unit)

E_pot =   -4.449999998344918 eV
A_surface = 3.0975990100882553 nm^2
E_surface_f = 1877.948373528271 mJ/m^2


In [15]:
# Show info for the base systems
print('E_total_base =', uc.get_in_units(results_dict['E_total_base'], energy_unit), energy_unit)
print('Dumped to', results_dict['dumpfile_base'])

E_total_base = -4449.999998344918 eV
Dumped to perfect.dump


In [16]:
# Show info for the base systems
print('E_total_surf =', uc.get_in_units(results_dict['E_total_surf'], energy_unit), energy_unit)
print('Dumped to', results_dict['dumpfile_surf'])

E_total_surf = -4377.384646212176 eV
Dumped to surface.dump


### 4.3. Optional: Clean calculation files

The calculation may generate output files when it runs.  Calling calculation.clean_files() will delete any generated files to keep the workspace clean.

In [17]:
calculation.clean_files()